**Components:**

**LLM** - Brain / Decision Maker (Ex: Groq's LLaMa model)

**Tools** - Functions it can call (Ex: SQL executer, web search)

**Memory** - Conversation History (Ex: Chat History)

**Reasoning** - Multi-step planning (Ex: Chain of thought)

In [ ]:
!pip install groq -q
print("libraries installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 8.0 MB/s eta 0:00:00
libraries installed successfully


In [ ]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("all the libraries are imported successfully")

all the libraries are imported successfully


api key used

In [ ]:
import os
os.environ["groq_api_key"]="xxxxxxxxxxxxxx"
client =Groq(api_key=os.environ["groq_api_key"])
Model="llama-3.1-8b-instant"
print("groq client initiated successfully")
print("Using Model: ",Model)

groq client initiated successfully
Using Model:  llama-3.1-8b-instant


In [ ]:
import io
csv_data="""student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))
print(f"Dataset loaded : {len(df)} rows, {len(df.columns)} columns")


Dataset loaded : 30 rows, 8 columns


In [ ]:
#first 5 rows
df.head()

,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
#create sqlite databse and load the data
conn = sqlite3.connect("college.db")
df.to_sql("students", conn, if_exists="replace", index=False)
print("Database created and data loaded successfully")
print("Table 'students' created with 30 student records")

test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students", conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")

Database created and data loaded successfully
Table 'students' created with 30 student records

Verification: 30 rows in database


In [ ]:
def get_schema(conn, table_name='students'):
  cursor =conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines =[f"table : {table_name}"]
  schema_lines.append("Columns: ")
  for col in columns:
    schema_lines.append(f" - {col[1]} ({col[2]})")
    cursor.exeute(f"Select * from {table_name} limit 3")
    sample_rows=cursor.fetchall()
    schema_lines.append("\nSample rows (first 3):")
    for row in sample_rows:
      schema_lines.append(f" {row}")
    return "\n".join(schema_lines)

In [ ]:
def get_schema(conn, table_name='students'):
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"table : {table_name}"]
  schema_lines.append("Columns: ")
  for col in columns:
    schema_lines.append(f" - {col[1]} ({col[2]})")

  # Fetch sample rows once after defining all columns
  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_rows = cursor.fetchall()
  schema_lines.append("\nSample rows (first 3):")
  for row in sample_rows:
    schema_lines.append(f"  {row}")
  return "\n".join(schema_lines)

schema=get_schema(conn)
print(schema)

table : students
Columns: 
 - student_id (INTEGER)
 - name (TEXT)
 - age (INTEGER)
 - gender (TEXT)
 - subject (TEXT)
 - marks (INTEGER)
 - attendance (INTEGER)
 - grade (TEXT)

Sample rows (first 3):
  (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
  (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
  (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):
  system_prompt =f"""You are an expert SQL assistent.
  You are connected to a SQLite database with the following structure:

  {schema_text}

  Rules you must follow:
  1.Generate ONLY a valid SQLite SQL query.
  2.Do not include any explanation or text - only the SQL query.
  3.Do not use markdown code blocks. Return the raw SQL only.
  4.The table name is : students
  5.Only use column names that exist in the schema above.
  6.Use single quotes for string values in where clauses (example : where subject = 'Programming')
  7.If the user asks for top N, use order by marks desc limit N.
"""

  response =client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ], temperature=0.0
  )
  sql_query = response.choices[0].message.content.strip()
  return sql_query

  question ="Show me  all female students"
  print(f"Question: {question}")
  print("\nGenerating SQL...")

  sql=generate_sql(question,schema,client,Model)
  print(f"Generate SQL :\n {sql}")

In [ ]:
#Function to clean and execute the generated SQL query
def execute_sql(sql_query_string, conn):
  clean_sql = sql_query_string.strip() # Initialize clean_sql first
  # Remove markdown code block fences if present
  clean_sql = re.sub(r'```sql\s*', '', clean_sql)
  clean_sql = re.sub(r'```', '', clean_sql).strip()
  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None # Return df and no error
  except Exception as e:
    return None, str(e) # Return None for df and the error message

In [ ]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  """
  the main ai agent function,
  takes a user question in plain english and returns database results
  """
  print("="*60)
  print("user question: ",user_question)
  print("="*6)

  #step1:get the database schema
  if verbose:
    print("[step 1]:reading database schema .....")
  schema_text=get_schema(conn)
  if verbose:
    print("schema loaded sucessfully")
  if verbose:
    print("[step 2]:generating sql query.....")

  generated_sql=generate_sql(user_question,schema_text,client,model)
  if verbose:
    print("generated sql: ",generated_sql)
  if verbose:
    print("[step 3]:executing sql on the database...")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print("sql execution error: ",error)
    return None,generated_sql
  if verbose:
    print("[step 4] query returned",len(result_df),"rows")
    print("\nresult: ")
    print("-"*80)
    print(result_df.to_string(index=False))
    print("-"*80)
    return result_df,generated_sql
result,sql_used=text_to_sql_agent(
    "show top 5 students in programming",
    conn,client,Model
)

user question:  show top 5 students in programming
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
        name
Aditya Kumar
 Rohan Mehta
Anjali Singh
 Nandita Rao
  Arjun Nair
--------------------------------------------------------------------------------


In [ ]:
result1, _ = text_to_sql_agent(
    "Show mw all students who study Mathematics",
    conn,
    client,
    Model
)

user question:  Show mw all students who study Mathematics
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE subject = 'Mathematics'
[step 3]:executing sql on the database...
[step 4] query returned 10 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
       

In [ ]:
result2,_=text_to_sql_agent(
    "What is the average mark for each subject?",
    conn,client,Model
)

user question:  What is the average mark for each subject?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT subject, AVG(marks) FROM students GROUP BY subject
[step 3]:executing sql on the database...
[step 4] query returned 3 rows

result: 
--------------------------------------------------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4
--------------------------------------------------------------------------------


In [ ]:
result3,_=text_to_sql_agent(
    "Show students who score more than 90 marks",
    conn,client,Model
)

user question:  Show students who score more than 90 marks
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE marks > 90
[step 3]:executing sql on the database...
[step 4] query returned 6 rows

result: 
--------------------------------------------------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+
--------------------------------------------------------------------------------


In [ ]:
result4,_=text_to_sql_agent(
    "How many male and female students are there?",
    conn,client,Model
)

user question:  How many male and female students are there?
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students
[step 3]:executing sql on the database...
[step 4] query returned 1 rows

result: 
--------------------------------------------------------------------------------
 male_count  female_count
         15            15
--------------------------------------------------------------------------------


In [ ]:
result5,_=text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn,client,Model
)

user question:  Show female students who scored above 85 in Science or Programming, ordered by marks
[step 1]:reading database schema .....
schema loaded sucessfully
[step 2]:generating sql query.....
generated sql:  SELECT * FROM students WHERE gender = 'Female' AND subject IN ('Science', 'Programming') AND marks > 85 ORDER BY marks DESC
[step 3]:executing sql on the database...
[step 4] query returned 5 rows

result: 
--------------------------------------------------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A
---------------------------------------------

In [ ]:
def generate_answer(user_question,query_results_df,client,model):
  if query_results_df is None or len(query_results_df)==0:
    return "No results were found for your query."
  results_text = query_results_df.to_string(index=False)
  prompt = f""" The user asked : '{user_question}'
  the databse returned these results:
  {results_text}
  Please write a clear , friendly, 2-3 sentence answer to the user question based on the results.
  Be specific . Mention actual names and numbers from the data.
  Do not add information not present in the results."""
  #call the llm with results
  response=client.chat.completions.create(
    model=model,
    messages=[{"role" : "user", "content":prompt}],
    temperature=0.3
)
  return response.choices[0].message.content.strip()



In [ ]:
#build enhanced agent with natural language response
def smart_text_to_sql_agent(user_question, conn, client, model):
  print()
  print("Question :", user_question)
  print()

  #get schema
  schema_text=get_schema(conn)
  #generate sql
  print("Generating sql...")
  generated_sql=generate_sql(user_question,schema_text,client,model)
  print("Generated SQL: ",generated_sql)
  #Execute sql
  result_df,error=execute_sql(generated_sql, conn)

  if error:
    print(f"Error execting SQL: {error}")
    return
  #show data table
  print(f"Query returned {len(result_df)} rows returned")
  #generate natural language answer
  print("\nGenerating natural language answer...")
  answer=generate_answer(user_question,result_df,client,model)
  print("Answer: ",answer)


In [ ]:
smart_text_to_sql_agent(
    "Who are the top 5 students in programming?",
    conn,client, Model
)


Question : Who are the top 5 students in programming?

Generating sql...
Generated SQL:  SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5
Query returned 5 rows returned

Generating natural language answer...
Answer:  Based on our database, the top 5 students in programming are: 

1. Aditya Kumar, 
2. Rohan Mehta, 
3. Anjali Singh, 
4. Nandita Rao, 
5. Arjun Nair.
